# 2. Data Preprocessing & Feature Engineering
## ICU XAI Research - Cleaning and Feature Creation

This notebook handles:
- Missing value imputation
- Outlier detection and handling
- Feature scaling/normalization
- Feature engineering for temporal data
- Train/val/test split

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from load_data import load_raw_data
from preprocess import ICUPreprocessor


In [ ]:
# Load raw set-a ICU data
data_dir = Path.cwd().parent / 'data'
df_raw = load_raw_data(data_dir, dataset='set-a', outcomes_file='Outcomes-a.txt')
print(f"Loaded raw data rows: {df_raw.shape[0]}")
print(f"Unique records: {df_raw['RecordID'].nunique()}")


In [ ]:
# Initialize preprocessor and apply the pipeline
preprocessor = ICUPreprocessor(test_size=0.2, val_size=0.1)
df_processed = preprocessor.preprocess(df_raw, timestep_minutes=60, window_hours=48)

print(f"Processed data shape: {df_processed.shape}")
print(df_processed.head(3))


In [ ]:
# Handle missing values
missing_summary = df_processed.isna().sum().sort_values(ascending=False).head(20)
print(missing_summary)


In [ ]:
# Feature engineering insights
print('Columns after preprocessing:', len(df_processed.columns))
print(df_processed.columns[:20].tolist())


In [ ]:
# Train/validation/test split
X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.split_data(df_processed, label_col='In-hospital_death')
print('Train:', X_train.shape, y_train.shape)
print('Val:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)


In [ ]:
# Save processed data to disk
data_processed_dir = data_dir / 'processed'
data_processed_dir.mkdir(parents=True, exist_ok=True)
df_processed.to_csv(data_processed_dir / 'processed_data.csv', index=False)
print(f"Saved processed data to {data_processed_dir / 'processed_data.csv'}")
